In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format='retina'
import daq
import spectrum as sp
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
df = daq.run_log()

df_ee_ae_s_filter = lambda ee, ae, s: df.loc[(df['excitation_energy_ev']==ee)&(df['arm_energy_ev']==ae)&(df['sample']==s)]

def df_y_filter(ee, ae, s, y, atol=1e-3):
    df = df_ee_ae_s_filter(ee, ae, s)
    return df.loc[(df['y'] - y).abs() < atol]

def df_T_filter(ee, ae, s, T, atol=5):
    df = df_ee_ae_s_filter(ee, ae, s)
    return df.loc[(df['temperature_k'] - T).abs() < atol]

In [ ]:
runs_1 = df_ee_ae_s_filter(533.96, 535, "vibrations").acq.to_numpy()
runs_2 = df_ee_ae_s_filter(538.96, 535, "vibrations").acq.to_numpy()
total_runs = [runs_1, runs_2]
for idx, runs in enumerate(total_runs):
    acqs = [daq.load_acq(a, calibrated=True, dark_correct=True).binned(8) for a in runs]
    acq = sp.combine(acqs).smooth_gauss(0.01)
    # l = f"Excitation Energy = {acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    l = f"acq({[int(a) for a in runs]}) @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    if idx == 1:
        plt.plot(acq.energy - 5, acq.intensity, label=l)
    else:
        acq.plot(label=l)
    print(acq.meta['counts'])
plt.legend(frameon=False)
# plt.ylim(0, 0.5)
plt.xlim(530, 535)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

In [ ]:
runs_1 = df_ee_ae_s_filter(533.96, 525, "water vapor").acq.to_numpy()
runs_2 = df_ee_ae_s_filter(538.96, 525, "water vapor").acq.to_numpy()
runs_3 = df_ee_ae_s_filter(548.96, 525, "water vapor").acq.to_numpy()
runs_4 = df_ee_ae_s_filter(533.96, 525, "jet").acq.to_numpy()
total_runs = [runs_1, runs_2, runs_3, runs_4]
for idx, runs in enumerate(total_runs):
    acqs = [daq.load_acq(a, calibrated=True, dark_correct=True).binned(8) for a in runs]
    acq = sp.combine(acqs).smooth_gauss(0.006).normalised()
    label_run_list = f"{[int(a) for a in runs]}" if len(runs) <= 2 else f"[{runs[0]} ... {runs[-1]}]"
    l = f"acq({label_run_list}) @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    print(acq.meta['counts'])
    acq.plot(label=l)
plt.legend(frameon=False)
# plt.ylim(0, 0.5)
plt.xlim(522, 526)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()

In [ ]:
runs_at_energy = df_ee_ae_s_filter(548.96, 525, "jet")
grouped_runs = runs_at_energy.groupby(runs_at_energy['temperature_k'])
colors = plt.get_cmap("viridis")(np.linspace(0, 0.9, len(grouped_runs)))
for idx, (name, group) in enumerate(grouped_runs):
    runs_in_group = group['acq'].to_numpy()
    acqs = [daq.load_acq(a, calibrated=True, dark_correct=True) for a in runs_in_group]
    acq = sp.combine(acqs).smooth_gauss(0.05).normalised(energy_range=(526, 529))
    label_run_list = f"{[int(a) for a in runs_in_group]}" if len(runs_in_group) <= 2 else f"[{runs_in_group[0]} ... {runs_in_group[-1]}]"
    l = f"acq({label_run_list}), T={group['temperature_k'].values[0]:.0f}K @ E={acq.meta['parts'][0]['beamline_energy']:.2f}eV ({len(acq.meta['parts'])} runs)"
    acq.plot(label=l, color=colors[idx])
plt.legend(frameon=False)
plt.xlim(523, 526)
plt.ylabel("Intensity (a.u.)")
plt.xlabel(r"Energy (eV)")
plt.show()